# PV Module Health — a Multimodal Case Study (I-V + EL)

Fuse two data modalities — electrical **I-V measurements** and visual
**electroluminescence (EL) images** — into one interpretable model of module
health under mechanical load.

**Unit of analysis:** one module at one exposure step (`sample_id` x
`exposure_step`). Four modules are stressed 0 → up to 5400 Pa; at each step we
have a paired EL image and I-V measurement.

> **Save a copy first:** `File -> Save a copy in Drive`.
> For the live segmentation, enable a GPU: `Runtime -> Change runtime type -> T4 GPU`
> (optional — everything else works on CPU, and results are precomputed).

## Section 0 — Setup

In [ ]:
import os, sys

REPO = "2026-data-science-pv-workshop"
RELEASE = "v0.1.0"   # pin code + data to a release for reproducibility

if not os.path.isdir(REPO) and os.path.basename(os.getcwd()) != REPO:
    !git clone --branch $RELEASE https://github.com/ucf-photovoltaics/{REPO}.git
if os.path.basename(os.getcwd()) != REPO:
    os.chdir(REPO)
sys.path.insert(0, "src")

# On Colab, torch / torchvision / opencv / pandas / sklearn are preinstalled.
from scripts.download_workshop_data import download_data
from scripts.download_weights import download_weights

download_data()      # EL images  -> data/workshop/images/01_input/  (~177 MB)
download_weights()   # model_97.pth -> data/workshop/models/          (~336 MB)

import torch
print("GPU available:", torch.cuda.is_available())

## Section 1 — Problem Formulation

**Question:** can visual defect features from EL images predict the electrical
performance loss measured by I-V?

- **Target:** `power_loss_percent` (Pmp drop vs the module's initial Pmp).
- **Unit:** one `(sample_id, exposure_step)` row.
- **"Good enough":** an interpretable model whose most important EL feature is
  physically sensible (crack extent) and that tracks the measured power loss.

## Section 2 — Physical & Digital Workflows

Two measurement streams meet at the shared `metadata.csv`:

- **I-V stream:** flash test -> Isc, Voc, Pmp, FF, Rs, Rsh (already in `metadata.csv`).
- **EL stream:** module image -> a 5-stage pipeline ->
  `01_input` -> `02_module_rectified` -> `03_cells` -> `04_cell_masks`
  -> `05_module_restitched`, then aggregated to per-image defect features.

The EL feature extraction is the `rdstemplate.el` package; the I-V features are
provided. We merge them on the EL image filename.

## Section 3 — Data Acquisition *(hands-on)*

In [ ]:
import pandas as pd, glob, os, cv2, matplotlib.pyplot as plt

meta = pd.read_csv("data/workshop/metadata.csv")
print("rows:", len(meta), "| modules:", meta.sample_id.nunique())
meta[["sample_id", "exposure_step", "pressure_pa", "iv_pmp_w",
      "power_loss_percent", "el_image_filename"]].head()

In [ ]:
# Look at one EL image (lowest pressure = baseline).
f = sorted(glob.glob("data/workshop/images/01_input/H5NE30994/*_0Pa*"))[0]
img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
plt.figure(figsize=(9, 6)); plt.imshow(img, cmap="gray"); plt.axis("off")
plt.title(os.path.basename(f)); plt.show()

## Section 4 — Pre-Processing & Feature Extraction *(hands-on)*

**I-V branch:** already reduced to parameters in `metadata.csv` (Isc, Voc, Pmp, ...).

**EL branch:** run the pipeline live on one image. On GPU this is seconds; on CPU
one image (~60 cells) takes ~30 s. (The *full* dataset is precomputed — Section 5.)

In [ ]:
from rdstemplate.el import rectify, cells, segment
import glob, os, cv2, matplotlib.pyplot as plt

serial = "2617127550699"   # Axitec — cracks badly under load
img = sorted(glob.glob(f"data/workshop/images/01_input/{serial}/*3600PA*"))[0]

os.makedirs("tmp/cells", exist_ok=True)
rectify.rectify_file(img, "tmp/rect.png")                 # stage 2
rows, ncols = cells.GRID[serial]
cells.crop_file("tmp/rect.png", "tmp/cells", rows, ncols) # stage 3

model = segment.load_model("data/workshop/models/model_97.pth")  # stage 4
tiles = []
for cf in sorted(glob.glob("tmp/cells/cell_r01_c*.png")):
    g = cv2.imread(cf, cv2.IMREAD_GRAYSCALE)
    m = segment.predict_mask(model, g)
    tiles.append(cv2.cvtColor(segment.overlay_mask(g, m), cv2.COLOR_BGR2RGB))

plt.figure(figsize=(16, 2))
for i, t in enumerate(tiles):
    plt.subplot(1, len(tiles), i + 1); plt.imshow(t); plt.axis("off")
plt.suptitle("Row 1 cells at 3600 Pa — red = crack, blue = contact"); plt.show()

**Pitfalls to discuss:** cell-grid must match the module (60 vs 72 vs 96 cells);
the mask threshold trades sensitivity vs false positives; aggregating cell
fractions to a module feature discards *where* the damage is.

## Section 5 — Data Fusion, Modeling & Analysis *(hands-on)*

We use the **precomputed** merged table (`metadata_with_el_features.csv` = I-V +
EL features for all rows) so this section runs for everyone regardless of GPU.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv("data/workshop/metadata_with_el_features.csv")
feat_cols = ["el_crack_frac_mean", "el_defect_frac_total",
             "el_frac_cells_cracked", "el_crack_frac_max"]
d = df.dropna(subset=feat_cols + ["power_loss_percent"])
X, y = d[feat_cols], d["power_loss_percent"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
rf = RandomForestRegressor(n_estimators=200, random_state=0).fit(Xtr, ytr)
pred = rf.predict(Xte)
print("R2:", round(r2_score(yte, pred), 3),
      "| MAE:", round(mean_absolute_error(yte, pred), 2), "%")
print("\nFeature importances:")
for c, imp in sorted(zip(feat_cols, rf.feature_importances_), key=lambda x: -x[1]):
    print(f"  {c:24s} {imp:.3f}")

**Validation caveats (discuss!):** only 4 modules and ~100 rows. A random split
puts different pressure steps of the *same* module in both train and test —
optimistic (leakage). A stricter test groups by `sample_id` (hold out whole
modules); with only 4 modules the score gets noisy, which is the honest small-
sample story. Report uncertainty, not just a point R2.

## Section 6 — Insights & Interpretation

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv("data/workshop/metadata_with_el_features.csv").dropna(subset=["el_crack_frac_mean"])
plt.figure(figsize=(7, 5))
for sid, g in df.groupby("sample_id"):
    plt.scatter(g["el_crack_frac_mean"], g["power_loss_percent"], label=sid, alpha=0.8)
plt.xlabel("EL mean crack fraction"); plt.ylabel("I-V power loss (%)")
plt.title("EL defect signal vs measured power loss"); plt.legend(); plt.show()

The EL crack fraction rises with load and tracks I-V power loss — a genuine
multimodal signal. **Correlation, not proof of mechanism:** cracks isolate cell
area (less current) which lowers Pmp, but other losses (Rs, interconnect) also
contribute. Interpret feature importance in that light.

## Section 7 — Iterative Improvements

- **Spatial features:** where cracks sit (edge vs center, contiguous inactive
  area) likely predicts loss better than a single mean fraction.
- **Leakage-aware validation:** group-by-module CV; report intervals.
- **More modules:** 4 is tiny — the model is a teaching artifact, not a
  deployable predictor.
- **Stretch:** train a small CNN on EL cells directly instead of hand-aggregated
  fractions, and compare.